<a href="https://colab.research.google.com/github/Western-Windows/SociaLift/blob/engagement/engagement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0. Install dependencies (Colab / local with internet)
%pip install -q transformers datasets accelerate sentencepiece scikit-learn torch tqdm

In [ ]:
# 1. Imports and configuration

import json
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    get_linear_schedule_with_warmup
)

tqdm.pandas()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
# 2. Load JSON and flatten to DataFrame

with open("SociaLift_Scraped_Data.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Assume raw_data is a list of posts; adjust if nested
df = pd.json_normalize(raw_data)

print(df.head())
print(df.columns)

      brand  username                                              posts  \
0    ADIDAS    adidas  [{'post_text': 'The new standard for Hybrid Fi...   
1      Nike      nike  [{'post_text': 'Leave your limits at the surfa...   
2      Puma      PUMA  [{'post_text': 'DEVIATE ELITE HYROX. Made for ...   
3  Converse  Converse  [{'post_text': 'Spotted: @lilamoss in the new ...   
4    Reebok    Reebok  [{'post_text': 'BORN CLASSIC. WORN FOR LIFE. S...   

  stats.original_json_followers stats.scraped_followers  
0                           43M                     43M  
1                           39M                     39M  
2                          ~23M                     23M  
3                          ~46M                     46M  
4                          ~10M                     10M  
Index(['brand', 'username', 'posts', 'stats.original_json_followers',
       'stats.scraped_followers'],
      dtype='object')


In [ ]:
# 3. Build total_reacts and engagement_class (low / avg / high)

# The original df has a 'posts' column which is a list of dictionaries.
# We need to flatten this structure to have one row per post.
flattened_posts_data = []

# Iterate over the original DataFrame rows (each row represents a brand/user)
for index, row in df.iterrows():
    brand_info = {
        "brand": row["brand"],
        "username": row["username"],
        "stats.original_json_followers": row["stats.original_json_followers"],
        "stats.scraped_followers": row["stats.scraped_followers"],
    }

    # Iterate over the list of posts for the current brand/user
    if "posts" in row and isinstance(row["posts"], list):
        for post_dict in row["posts"]:
            # Combine brand info with post-specific info
            full_post_data = brand_info.copy()
            full_post_data.update(post_dict)
            flattened_posts_data.append(full_post_data)

# Create a new DataFrame from the flattened data
df = pd.DataFrame(flattened_posts_data)

# Now, we should have 'post_text' and 'reactions'/'total_reactions' directly in the df.
# Let's redefine get_total_reacts as it was written to handle a row,
# but now the 'row' itself will be a single post, not a brand with a list of posts.

def get_total_reacts(row):
    if "total_reactions" in row and pd.notna(row["total_reactions"]):
        return row["total_reactions"]
    if "reactions" in row and isinstance(row["reactions"], dict):
        if "total" in row["reactions"]:
            return row["reactions"]["total"]
        # Sum all reaction types if 'total' is not present
        return sum(v for k, v in row["reactions"].items() if isinstance(v, (int, float)))
    # If no reaction data is found, return NaN to be dropped later
    return np.nan

text_col_candidates = ["post_text", "text", "content"]
post_text_col = None
for c in text_col_candidates:
    if c in df.columns:
        post_text_col = c
        break
# After flattening, we expect 'post_text' to be present.
if post_text_col is None:
    # This should ideally not happen after the flattening above if 'post_text' exists in the raw data
    # but keep it as a safeguard.
    raise ValueError("No suitable text column found after flattening 'posts' column; please check data structure.")

df["total_reacts_num"] = df.apply(get_total_reacts, axis=1)

# Drop rows where 'post_text_col' or 'total_reacts_num' is missing
df = df.dropna(subset=[post_text_col, "total_reacts_num"]).copy()
df["total_reacts_num"] = df["total_reacts_num"].astype(float)

# Calculate quantiles for engagement classification
# Check if df is not empty before calculating quantiles
if not df.empty:
    q1 = df["total_reacts_num"].quantile(1.0 / 3.0)
    q2 = df["total_reacts_num"].quantile(2.0 / 3.0)
else:
    q1 = 0
    q2 = 0 # Handle empty df case

def label_engagement(v):
    if v <= q1:
        return "low"
    elif v <= q2:
        return "avg"
    else:
        return "high"

df["engagement_class"] = df["total_reacts_num"].apply(label_engagement)
df["post_text_en"] = df[post_text_col].astype(str)

print(df[[post_text_col, "post_text_en", "total_reacts_num", "engagement_class"]].head())
print(df["engagement_class"].value_counts(normalize=True))

                                           post_text  \
0  The new standard for Hybrid Fitness Racing. AD...   
1       Leave your limits at the surface.  #JustDoIt   
2  The longest jump is the one you didn’t doubt. ...   
3               Two thoughts, you’re out.  #justdoit   
4                                  Opposites attack.   

                                        post_text_en  total_reacts_num  \
0  The new standard for Hybrid Fitness Racing. AD...             709.0   
1       Leave your limits at the surface.  #JustDoIt            4001.0   
2  The longest jump is the one you didn’t doubt. ...            2682.0   
3               Two thoughts, you’re out.  #justdoit            1949.0   
4                                  Opposites attack.            5315.0   

  engagement_class  
0             high  
1             high  
2             high  
3             high  
4             high  
engagement_class
high    0.343284
avg     0.328358
low     0.328358
Name: proportion, dtype:

In [ ]:
# 4. Optional translation (only run if you *definitely* need it)

translate = True  # set to True if you want this

if translate:
    translation_model_name = "Helsinki-NLP/opus-mt-mul-en"
    trans_tokenizer = AutoTokenizer.from_pretrained(translation_model_name)
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(translation_model_name).to(device)

    def translate_batch(texts, max_length=128, batch_size=16):
        out = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
            batch = texts[i:i+batch_size]
            enc = trans_tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length
            ).to(device)
            with torch.no_grad():
                gen = trans_model.generate(**enc, max_length=max_length)
            decoded = trans_tokenizer.batch_decode(gen, skip_special_tokens=True)
            out.extend(decoded)
        return out

    df["post_text_en"] = translate_batch(df[post_text_col].tolist())

In [ ]:
# 5. Train/validation split and label encoding

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["engagement_class"])
num_labels = len(label_encoder.classes_)
print("Labels:", list(label_encoder.classes_), "num_labels:", num_labels)

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

print("Train size:", len(train_df), "Val size:", len(val_df))

Labels: ['avg', 'high', 'low'] num_labels: 3
Train size: 53 Val size: 14


In [ ]:
# 6. TF-IDF + Logistic Regression baseline

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3
)

X_train_tfidf = tfidf.fit_transform(train_df["post_text_en"])
X_val_tfidf = tfidf.transform(val_df["post_text_en"])

log_reg = LogisticRegression(
    max_iter=200,
    n_jobs=-1,
    class_weight="balanced"
)
log_reg.fit(X_train_tfidf, train_df["label_id"])

val_pred_lr = log_reg.predict(X_val_tfidf)
val_pred_lr_proba = log_reg.predict_proba(X_val_tfidf)

print("TF-IDF + LR accuracy:", accuracy_score(val_df["label_id"], val_pred_lr))
print("TF-IDF + LR macro F1:", f1_score(val_df["label_id"], val_pred_lr, average="macro"))
print(classification_report(val_df["label_id"], val_pred_lr, target_names=label_encoder.classes_))

TF-IDF + LR accuracy: 0.5714285714285714
TF-IDF + LR macro F1: 0.5722943722943722
              precision    recall  f1-score   support

         avg       0.60      0.60      0.60         5
        high       1.00      0.40      0.57         5
         low       0.43      0.75      0.55         4

    accuracy                           0.57        14
   macro avg       0.68      0.58      0.57        14
weighted avg       0.69      0.57      0.57        14



In [ ]:
# 7. Torch dataset for transformer models

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": int(self.labels[idx])
        }

In [ ]:
# 8. Generic training utilities (works for BERT and RoBERTa models)

def collate_fn_builder(tokenizer, max_length=128):
    def collate(batch):
        texts = [item["text"] for item in batch]
        labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc["labels"] = labels
        return enc
    return collate

def train_transformer_classifier(
    model,
    tokenizer,
    train_df,
    val_df,
    num_epochs=3,
    batch_size=16,
    max_length=128,
    lr=2e-5,
    warmup_ratio=0.1
):
    train_dataset = TextDataset(train_df["post_text_en"], train_df["label_id"])
    val_dataset = TextDataset(val_df["post_text_en"], val_df["label_id"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn_builder(tokenizer, max_length)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn_builder(tokenizer, max_length)
    )

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    num_training_steps = num_epochs * len(train_loader)
    num_warmup_steps = int(warmup_ratio * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    best_f1 = 0.0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - train"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        model.eval()
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - val"):
                labels = batch["labels"].numpy()
                all_labels.extend(labels)

                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                all_preds.extend(preds)

        f1 = f1_score(all_labels, all_preds, average="macro")
        acc = accuracy_score(all_labels, all_preds)
        print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, val_acc={acc:.4f}, val_macro_f1={f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return model, best_f1

In [ ]:
# 9. BERT-base classifier with explicit softmax on top

from transformers import AutoModelForSequenceClassification, AutoTokenizer

bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    bert_model_name,
    num_labels=num_labels
)

bert_model, bert_best_f1 = train_transformer_classifier(
    bert_model,
    bert_tokenizer,
    train_df,
    val_df,
    num_epochs=3,
    batch_size=16,
    max_length=128,
    lr=2e-5
)

def bert_predict_with_softmax(texts):
    bert_model.eval()
    bert_model.to(device)
    enc = bert_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = bert_model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=1)
    labels = label_encoder.inverse_transform(preds)
    return labels, probs

val_labels = val_df["label_id"].values
_, bert_probs = bert_predict_with_softmax(val_df["post_text_en"].tolist())
bert_preds = np.argmax(bert_probs, axis=1)

print("BERT accuracy:", accuracy_score(val_labels, bert_preds))
print("BERT macro F1:", f1_score(val_labels, bert_preds, average="macro"))
print(classification_report(val_labels, bert_preds, target_names=label_encoder.classes_))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch 1: train_loss=1.1187, val_acc=0.5000, val_macro_f1=0.4051


Epoch 2/3 - val: 100%|██████████| 1/1 [00:06<00:00,  6.35s/it]


Epoch 2: train_loss=1.0734, val_acc=0.4286, val_macro_f1=0.3485


Epoch 3/3 - val: 100%|██████████| 1/1 [00:06<00:00,  6.35s/it]


Epoch 3: train_loss=1.0301, val_acc=0.4286, val_macro_f1=0.3636
BERT accuracy: 0.42857142857142855
BERT macro F1: 0.3636363636363636
              precision    recall  f1-score   support

         avg       0.50      0.60      0.55         5
        high       0.00      0.00      0.00         5
         low       0.43      0.75      0.55         4

    accuracy                           0.43        14
   macro avg       0.31      0.45      0.36        14
weighted avg       0.30      0.43      0.35        14



In [ ]:
# 10. RoBERTa-base classifier with explicit softmax on top

roberta_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_model_name,
    num_labels=num_labels
)

roberta_model, roberta_best_f1 = train_transformer_classifier(
    roberta_model,
    roberta_tokenizer,
    train_df,
    val_df,
    num_epochs=3,
    batch_size=16,
    max_length=128,
    lr=2e-5
)

def roberta_predict_with_softmax(texts):
    roberta_model.eval()
    roberta_model.to(device)
    enc = roberta_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = roberta_model(**enc)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=1)
    labels = label_encoder.inverse_transform(preds)
    return labels, probs

_, roberta_probs = roberta_predict_with_softmax(val_df["post_text_en"].tolist())
roberta_preds = np.argmax(roberta_probs, axis=1)

print("RoBERTa accuracy:", accuracy_score(val_labels, roberta_preds))
print("RoBERTa macro F1:", f1_score(val_labels, roberta_preds, average="macro"))
print(classification_report(val_labels, roberta_preds, target_names=label_encoder.classes_))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Epoch 2/3 - val: 100%|██████████| 1/1 [00:06<00:00,  6.34s/it]


Epoch 2: train_loss=1.1128, val_acc=0.2857, val_macro_f1=0.1481


Epoch 3/3 - val: 100%|██████████| 1/1 [00:05<00:00,  5.09s/it]


Epoch 3: train_loss=1.0928, val_acc=0.2857, val_macro_f1=0.1481
RoBERTa accuracy: 0.2857142857142857
RoBERTa macro F1: 0.14814814814814814
              precision    recall  f1-score   support

         avg       0.00      0.00      0.00         5
        high       0.00      0.00      0.00         5
         low       0.29      1.00      0.44         4

    accuracy                           0.29        14
   macro avg       0.10      0.33      0.15        14
weighted avg       0.08      0.29      0.13        14



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 11. Simple ensemble: average BERT and RoBERTa probabilities

ensemble_probs = (bert_probs + roberta_probs) / 2.0
ensemble_preds = np.argmax(ensemble_probs, axis=1)

print("Ensemble accuracy:", accuracy_score(val_labels, ensemble_preds))
print("Ensemble macro F1:", f1_score(val_labels, ensemble_preds, average="macro"))
print(classification_report(val_labels, ensemble_preds, target_names=label_encoder.classes_))

Ensemble accuracy: 0.42857142857142855
Ensemble macro F1: 0.42063492063492064
              precision    recall  f1-score   support

         avg       1.00      0.20      0.33         5
        high       0.67      0.40      0.50         5
         low       0.30      0.75      0.43         4

    accuracy                           0.43        14
   macro avg       0.66      0.45      0.42        14
weighted avg       0.68      0.43      0.42        14



In [ ]:
# 12. Collect metrics

results = []

# You might want to store these explicitly right after each training block;
# here is the pattern assuming you kept them.
tfidf_lr_macro_f1 = f1_score(val_df["label_id"], val_pred_lr, average="macro")
tfidf_lr_acc = accuracy_score(val_df["label_id"], val_pred_lr)

bert_macro_f1 = f1_score(val_labels, bert_preds, average="macro")
bert_acc = accuracy_score(val_labels, bert_preds)

roberta_macro_f1 = f1_score(val_labels, roberta_preds, average="macro")
roberta_acc = accuracy_score(val_labels, roberta_preds)

ensemble_macro_f1 = f1_score(val_labels, ensemble_preds, average="macro")
ensemble_acc = accuracy_score(val_labels, ensemble_preds)

results.append(["TF-IDF + LR", tfidf_lr_acc, tfidf_lr_macro_f1])
results.append(["BERT-base", bert_acc, bert_macro_f1])
results.append(["RoBERTa-base", roberta_acc, roberta_macro_f1])
results.append(["BERT + RoBERTa ensemble", ensemble_acc, ensemble_macro_f1])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Val Accuracy", "Val Macro F1"]
)
print(results_df)

                     Model  Val Accuracy  Val Macro F1
0              TF-IDF + LR      0.571429      0.572294
1                BERT-base      0.428571      0.363636
2             RoBERTa-base      0.285714      0.148148
3  BERT + RoBERTa ensemble      0.428571      0.420635
